# SymbioPan — Full Project Tour

This notebook imports and demonstrates **every module** in the SymbioPan codebase for the PUMA Grand Challenge Track 2 panoptic segmentation pipeline.

## Package Structure
- `configs/` — Centralized configuration dataclasses
- `data/` — Constants, preprocessing, dataset, transforms, sampling
- `models/` — Encoder, backbone, FPN, decoders, cross-attention, panoptic net, Stage 2 refiner
- `training/` — Train loop, checkpoint, logging, CLI, Stage 1 & 2 trainers
- `inference/` — WSI tiling, model loading, site classifier, cellpose flow, postprocessing
- `utils/` — Losses, metrics, spatial prior, SC-DFA, split utils, normalization
- `scripts/` — Entry-point scripts

In [ ]:
import sys, torch, numpy as np
import matplotlib.pyplot as plt
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---
## 0. Google Colab Setup

These cells set up the environment: mount Google Drive (for dataset access), clone the repo, and install dependencies.

In [ ]:
# Mount Google Drive to access dataset
from google.colab import drive
drive.mount('/content/drive')

# Create a symlink or adjust PATHS to point to your dataset on Drive
# Adjust this path to where your processed dataset is stored
import os
DRIVE_DATASET_PATH = "/content/drive/MyDrive/Research/dataset_processed"
if os.path.isdir(DRIVE_DATASET_PATH):
    print(f"Dataset found at {DRIVE_DATASET_PATH}")
else:
    print(f"Dataset NOT found at {DRIVE_DATASET_PATH}. Please update DRIVE_DATASET_PATH to your dataset location.")

In [ ]:
# Clone the SymbioPan repository (only if not already cloned)
import os
if not os.path.isdir("SymbioPan"):
    !git clone https://github.com/hoangtung386/SymbioPan.git
    %cd SymbioPan
else:
    %cd SymbioPan
    print("Repository already cloned. Pulling latest changes...")
    !git pull

In [ ]:
# Install dependencies via requirements.txt 
!pip install -qqq -r requirements.txt

In [ ]:
# Update PATHS to point to the dataset on Google Drive
from configs import PATHS
from pathlib import Path

# Override paths for Colab environment
if os.path.isdir(DRIVE_DATASET_PATH):
    object.__setattr__(PATHS, 'data_dir', Path(DRIVE_DATASET_PATH))
    print(f"PATHS.data_dir set to: {PATHS.data_dir}")
else:
    print(f"WARNING: Dataset not found. Using default path: {PATHS.data_dir}")
print(f"PATHS.checkpoint_dir: {PATHS.checkpoint_dir}")
print(f"PATHS.split_file: {PATHS.split_file}")

### GPU Auto-Configuration for High-VRAM Environment

Detects available GPU(s) and overrides configs to fully utilize 100GB VRAM (Colab Pro G4 / A100).

In [ ]:
import torch, gc
from configs import STAGE1_DEFAULT_CONFIG, STAGE2_DEFAULT_CONFIG

def detect_gpu_setup(force_batch_size=None):
    """Auto-detect GPU specs and override configs for maximum VRAM utilization."""
    if not torch.cuda.is_available():
        print("No GPU detected. Using CPU defaults.")
        return

    num_gpus = torch.cuda.device_count()
    vram_gb = []
    for i in range(num_gpus):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        vram_gb.append(vram)
        print(f"  GPU {i}: {name} | {vram:.1f} GB VRAM")

    total_vram = sum(vram_gb)
    peak_vram = max(vram_gb)
    print(f"\nDetected: {num_gpus} GPU(s), total VRAM = {total_vram:.1f} GB")

    # ---- Override configs based on VRAM budget ----
    if force_batch_size is not None:
        bs = force_batch_size
    elif peak_vram >= 75:
        bs = 64  # A100 80GB
    elif peak_vram >= 40:
        bs = 32  # A100 40GB / V100 32GB
    elif peak_vram >= 16:
        bs = 16
    else:
        bs = 8

    if num_gpus > 1:
        bs = bs * num_gpus

    print(f"  -> Stage 1 batch_size = {bs} (effective: {bs * max(num_gpus, 1)})")
    print(f"  -> Stage 2 batch_size = {min(bs * 2, 128)}")
    print(f"  -> num_workers = {min(8, num_gpus * 4)}")

    # Patch configs in-place (these are mutable dataclass instances)
    object.__setattr__(STAGE1_DEFAULT_CONFIG, 'batch_size', bs)
    object.__setattr__(STAGE1_DEFAULT_CONFIG, 'num_workers', min(8, num_gpus * 4))
    object.__setattr__(STAGE1_DEFAULT_CONFIG, 'multi_gpu', num_gpus > 1)
    object.__setattr__(STAGE1_DEFAULT_CONFIG, 'samples_per_epoch_multiplier', 2.0)
    object.__setattr__(STAGE1_DEFAULT_CONFIG, 'use_fp16', True)

    object.__setattr__(STAGE2_DEFAULT_CONFIG, 'batch_size', min(bs * 2, 128))
    object.__setattr__(STAGE2_DEFAULT_CONFIG, 'num_workers', min(8, num_gpus * 4))
    object.__setattr__(STAGE2_DEFAULT_CONFIG, 'samples_per_epoch_multiplier', 3.0)
    object.__setattr__(STAGE2_DEFAULT_CONFIG, 'use_fp16', True)

    # Enable bfloat16 if supported (A100, H100)
    if torch.cuda.is_bf16_supported():
        print("  -> bfloat16 supported: mixed precision will use bf16")
    else:
        print("  -> Using float16 mixed precision")

    # Clear GPU cache
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nGPU cache cleared. Available VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB")

detect_gpu_setup()

# Optionally force a specific batch size:
# detect_gpu_setup(force_batch_size=48)

### Enable bfloat16 for A100/H100

When available, use bfloat16 instead of float16 for more stable mixed-precision training on Ampere+ GPUs.

In [ ]:
from configs import STAGE1_DEFAULT_CONFIG as S1C, STAGE2_DEFAULT_CONFIG as S2C
from training.train_loop import validate, _batch_to_device, _autocast_context

# Patch the autocast context to use bfloat16 on Ampere+ GPUs
import training.train_loop as tl
original_autocast = tl._autocast_context

def _patched_autocast(device):
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return original_autocast(device)

tl._autocast_context = _patched_autocast
print(f"Using bfloat16: {torch.cuda.is_bf16_supported()}")

# Enable gradient checkpointing for ViT encoder to save memory for bigger batches
from models.encoder import UnifiedPanopticEncoder
def enable_grad_ckpt(model):
    if hasattr(model, 'encoder') and hasattr(model.encoder, 'vit_model'):
        vit = model.encoder.vit_model
        if hasattr(vit, 'gradient_checkpointing_enable'):
            vit.gradient_checkpointing_enable()
            print("ViT gradient checkpointing enabled.")

print("\nHigh-VRAM optimizations ready.")

---
## 1. configs/ — Configuration

In [ ]:
from configs import (
    PATHS, INFERENCE_DEFAULT_CONFIG, PREPROCESS_DEFAULT_CONFIG,
    STAGE1_DEFAULT_CONFIG, STAGE2_DEFAULT_CONFIG,
    InferenceConfig, PathsConfig, PreprocessConfig, Stage1Config, Stage2Config,
)
from configs.defaults import get_device, linear_ramp
from configs.serialization import make_inference_config_from_stage1

print("Paths:", PATHS)
print("Stage 1 batch_size:", STAGE1_DEFAULT_CONFIG.batch_size)
print("Preprocess image_size:", PREPROCESS_DEFAULT_CONFIG.image_size)
print("get_device():", get_device())
print("linear_ramp(epoch=12, start=10, end=16, max=0.5):", linear_ramp(12, 10, 16, 0.5))

### 1b. linear_ramp Schedule Visualization

Visualize how `linear_ramp` controls the smooth activation of focal loss, SC-DFA, and spatial prior during training.

In [ ]:
epochs = range(1, 51)
focal_w = [linear_ramp(e, STAGE1_DEFAULT_CONFIG.focal_start_epoch, STAGE1_DEFAULT_CONFIG.focal_full_epoch, STAGE1_DEFAULT_CONFIG.focal_max_weight) for e in epochs]
scdfa_w = [linear_ramp(e, STAGE1_DEFAULT_CONFIG.sc_dfa_start_epoch, STAGE1_DEFAULT_CONFIG.sc_dfa_full_epoch, STAGE1_DEFAULT_CONFIG.sc_dfa_max_weight) for e in epochs]
prior_w = [linear_ramp(e, STAGE1_DEFAULT_CONFIG.prior_start_epoch, STAGE1_DEFAULT_CONFIG.prior_full_epoch, STAGE1_DEFAULT_CONFIG.prior_max_weight) for e in epochs]

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(epochs, focal_w, label='FocalTversky weight', linewidth=2)
ax.plot(epochs, scdfa_w, label='SC-DFA lambda', linewidth=2)
ax.plot(epochs, prior_w, label='Spatial Prior lambda', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Weight / Lambda')
ax.set_title('Stage 1 Smooth Schedule (linear_ramp)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. data/ — Constants, Preprocessing, Dataset

### 2a. data.constants — Label mappings & weights

In [ ]:
from data.constants import (
    PUMA_TISSUE_ID_TO_NAME, INTERNAL_TISSUE_ID_TO_NAME, NUM_TISSUE_CLASSES,
    PUMA_NUCLEI_ID_TO_NAME, NUM_NUCLEI_CLASSES,
    RARE_TISSUE_IDS, RARE_NUCLEI_IDS,
    TISSUE_CLASS_WEIGHTS, NUCLEI_CLASS_WEIGHTS, STAGE2_NUCLEI_WEIGHTS,
    LOSS_MULTIPLIERS, HV_GRAD_THRESHOLD,
    NORMALIZATION_MEAN, NORMALIZATION_STD, IGNORE_INDEX,
)

print(f"Tissue classes ({NUM_TISSUE_CLASSES}):", INTERNAL_TISSUE_ID_TO_NAME)
print(f"Nuclei classes ({NUM_NUCLEI_CLASSES}):", PUMA_NUCLEI_ID_TO_NAME)
print(f"Tissue weights:", TISSUE_CLASS_WEIGHTS)
print(f"Nuclei weights:", NUCLEI_CLASS_WEIGHTS)
print(f"Normalization:", NORMALIZATION_MEAN, NORMALIZATION_STD)

### 2a-ii. Class-Weight Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

tissue_names = [INTERNAL_TISSUE_ID_TO_NAME[i] for i in range(NUM_TISSUE_CLASSES)]
axes[0].bar(tissue_names, TISSUE_CLASS_WEIGHTS, color=['#2196F3' if i not in RARE_TISSUE_IDS else '#F44336' for i in range(NUM_TISSUE_CLASSES)])
axes[0].set_title('Tissue Class Weights (red=rare)')
axes[0].tick_params(axis='x', rotation=30)

nuclei_names = [PUMA_NUCLEI_ID_TO_NAME[i] for i in range(NUM_NUCLEI_CLASSES)]
axes[1].bar(nuclei_names, NUCLEI_CLASS_WEIGHTS, color=['#2196F3' if i not in RARE_NUCLEI_IDS else '#F44336' for i in range(NUM_NUCLEI_CLASSES)])
axes[1].set_title('Stage 1 Nuclei Weights (red=rare)')
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(nuclei_names, STAGE2_NUCLEI_WEIGHTS, color=['#2196F3' if i not in RARE_NUCLEI_IDS else '#F44336' for i in range(NUM_NUCLEI_CLASSES)])
axes[2].set_title('Stage 2 Nuclei Weights (red=rare)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2b. data.preprocessing — GeoJSON parsing, flow generation

In [ ]:
from data.preprocessing.geojson_parser import parse_geojson_masks, find_annotation_file
from data.preprocessing.flow_generator import CellposeFlowGenerator, compute_hv_map, compute_hv_map_torch
from data.preprocessing import main as preprocess_main

# Demonstrate helper functions with dummy data
dummy_inst = np.zeros((64, 64), dtype=np.int32)
dummy_inst[10:20, 10:30] = 1
dummy_inst[40:50, 40:55] = 2
hv_map = compute_hv_map(dummy_inst)
print(f"compute_hv_map output shape: {hv_map.shape}")

dummy_tensor = torch.from_numpy(dummy_inst)
hv_torch = compute_hv_map_torch(dummy_tensor)
print(f"compute_hv_map_torch output shape: {hv_torch.shape}")

# CellposeFlowGenerator (disabled to avoid loading cellpose)
cp_gen = CellposeFlowGenerator(enabled=False, model_type="nuclei")
dummy_image = np.zeros((64, 64, 3), dtype=np.uint8)
flow = cp_gen.make_flow(dummy_image)
print(f"CellposeFlowGenerator (disabled) output shape: {flow.shape}")

### 2c. data.dataset — Dataset, transforms, sampling

In [ ]:
from data.dataset import PUMADataset, get_train_transforms, get_val_transforms
from data.dataset.puma_dataset import puma_tissue_to_internal, internal_tissue_to_puma, source_name_from_base
from data.dataset.sampling import compute_sample_weight, compute_all_sample_weights
from data.dataset.transforms import VectorSafeCompose

# Demonstrate label conversion
puma_labels = np.array([0, 1, 2, 3, 4, 5], dtype=np.uint8)
internal = puma_tissue_to_internal(puma_labels)
back_to_puma = internal_tissue_to_puma(internal)
print(f"PUMA -> internal: {puma_labels} -> {internal}")
print(f"internal -> PUMA: {internal} -> {back_to_puma}")

# Source name extraction
print(f"source_name_from_base('roi01__rare00_tissue2'): {source_name_from_base('roi01__rare00_tissue2')}")

# Transforms
train_tf = get_train_transforms(1024)
val_tf = get_val_transforms(1024)
print(f"Train transforms: {type(train_tf).__name__}")
print(f"Val transforms: {type(val_tf).__name__}")

# Demonstrate sample weighting for rare classes
tissue_with_rare = np.array([0, 0, 2, 4, 2, 2, 0, 1])  # PUMA ids; 4=rare necrosis, 1=rare blood vessel
nuclei_with_rare = np.array([0, 0, 2, 5, 4, 6, 0, 1])     # 2=rare plasma, 5=rare neutrophil, 4=rare melanophage
w_normal = compute_sample_weight(tissue_with_rare, nuclei_with_rare, is_rare_augmented=False)
w_augmented = compute_sample_weight(tissue_with_rare, nuclei_with_rare, is_rare_augmented=True)
print(f"\nSample weight (non-augmented): {w_normal:.2f}")
print(f"Sample weight (rare-augmented):  {w_augmented:.2f}")

### 2d. Try loading the processed dataset (if available)

In [ ]:
data_dir = PATHS.data_dir
try:
    ds = PUMADataset(data_dir, transforms=get_val_transforms(1024))
    print(f"Dataset loaded: {len(ds)} samples")
    batch = ds[0]
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: {v.shape}")
        else:
            print(f"  {k}: {v}")
except Exception as e:
    print(f"Dataset not available: {e}")

---
## 3. models/ — Neural Network Architectures

### 3a. backbone — ConvNeXt-Atto

In [ ]:
from models import build_cnn_backbone, UnifiedPanopticNet, ResidualNucleiRefinerUNet, build_stage2_input
from models.backbone import get_cnn_spatial_prior
from models.encoder import build_uni_vit, get_frozen_uni_model, UnifiedPanopticEncoder
from models.cross_attention import SpatialInjector
from models.fpn_aggregator import FPNAggregator
from models.decoders import (
    ParallelDecoders, MutualFeatureExchange,
    HoVerNeXtNucleiHead, ASPP, ASPPBranch,
)
from models.panoptic_net import UnifiedPanopticNet
from models.stage2_refiner import ResidualNucleiRefinerUNet, build_stage2_input

# Build CNN backbone
cnn = build_cnn_backbone(pretrained=False)
print(f"CNN backbone: {type(cnn).__name__}")
if hasattr(cnn, 'feature_info'):
    print(f"  feature channels: {cnn.feature_info.channels()}")

# FPN
fpn = FPNAggregator(vit_dim=1024, cnn_dims=[40, 80, 160, 320], fpn_dim=256)
x = torch.randn(1, 257, 1024)
cnn_feats = [torch.randn(1, d, 256//(2**i), 256//(2**i)) for i, d in enumerate([40, 80, 160, 320])]
fpn_out = fpn(x, cnn_feats)
print(f"FPN output keys: {list(fpn_out.keys())}")
for k, v in fpn_out.items():
    print(f"  {k}: {v.shape}")

# SpatialInjector
injector = SpatialInjector(vit_dim=1024, cnn_dims=[40, 80, 160, 320])
vit_tokens = torch.randn(1, 257, 1024)
cnn_feats = [torch.randn(1, d, 64, 64) for d in [40, 80, 160, 320]]
out = injector(vit_tokens, cnn_feats)
print(f"SpatialInjector output shape: {out.shape}")

# MutualFeatureExchange
mfe = MutualFeatureExchange(dim=256)
ft, fn = mfe(torch.randn(1, 256, 32, 32), torch.randn(1, 256, 32, 32))
print(f"MFE output shapes: ft={ft.shape}, fn={fn.shape}")

# ParallelDecoders
dec = ParallelDecoders(fpn_dim=256, num_tissue=5, num_nuclei=10)
fpn_feats = {k: torch.randn(1, 256, s, s) for k, s in zip(["p1","p2","p3","p4","p5"], [128, 64, 32, 16, 8])}
t, np_, nc, hv = dec(fpn_feats, torch.randn(1, 2, 128, 128))
print(f"Decoder outputs: tissue={t.shape}, np={np_.shape}, nc={nc.shape}, hv={hv.shape}")

# ASPP & HoVerNeXt head
aspp = ASPP(256, 256)
print(f"ASPP: {aspp(torch.randn(1, 256, 32, 32)).shape}")
hn = HoVerNeXtNucleiHead(258, 64, 2)
print(f"HoVerNeXt head: {hn(torch.randn(1, 258, 128, 128)).shape}")

### 3b. Stage 2 Refiner

In [ ]:
# ResidualNucleiRefinerUNet
s2 = ResidualNucleiRefinerUNet(in_channels=21, out_classes=10)
s2_in = torch.randn(1, 21, 128, 128)
s2_out = s2(s2_in)
print(f"Stage 2 refiner output shape: {s2_out.shape}")

# Verify zero-init
print(f"Final conv zero-init: weight sum = {s2.outc.weight.abs().sum().item():.6f}")

### 3c. Full UnifiedPanopticNet (Stage 1)

In [ ]:
# Build the full model (no UNI weights for demo)
cnn = build_cnn_backbone(pretrained=False)
model = UnifiedPanopticNet(
    vit_model=None,
    cnn_model=cnn,
    num_tissue=5,
    num_nuclei=10,
    load_uni_weights=False,
)

# Forward pass with dummy data
model.eval()
images = torch.randn(1, 3, 256, 256)
cp_flows = torch.randn(1, 2, 256, 256)
with torch.no_grad():
    out = model(images, cp_flows, site_types=None)
print("UnifiedPanopticNet outputs:")
for k, v in out.items():
    print(f"  {k}: {v.shape}")

# SC-DFA and spatial prior
print(f"\nSC-DFA available: {hasattr(model, 'sc_dfa')}")
print(f"Spatial prior available: {hasattr(model, 'spatial_prior')}")
model.enable_sc_dfa(True)
model.set_sc_dfa_lambda(0.3)
model.set_spatial_prior_lambda(0.2)
print(f"SC-DFA lambda: {model.lambda_sc_dfa}, Prior lambda: {model.lambda_prior}")

# Demonstrate make_inference_config_from_stage1
infer_cfg = make_inference_config_from_stage1(STAGE1_DEFAULT_CONFIG, model)
print(f"\nInference config keys: {list(infer_cfg.keys())}")
print(f"  architecture: {infer_cfg['architecture']}")
print(f"  use_sc_dfa: {infer_cfg['use_sc_dfa']}")
print(f"  lambda_sc_dfa: {infer_cfg['lambda_sc_dfa']}")
print(f"  lambda_prior: {infer_cfg['lambda_prior']}")

---
## 4. training/ — Training Pipeline

In [ ]:
from training import (
    stage1_main, stage2_main,
    train_one_epoch, validate,
    safe_torch_save, load_large_checkpoint, extract_state_dict,
    logger, setup_logger,
)
from training.cli import parse_stage1_args, parse_stage2_args
from training.train_loop import _batch_to_device, _autocast_context
from training.checkpoint import safe_torch_save, load_large_checkpoint, extract_state_dict
from training.stage1_trainer import apply_smooth_schedule, make_rare_weighted_sampler, print_report as stage1_print_report
from training.stage2_trainer import compute_stage2_scores, alpha_schedule, keep_lambda_schedule

print(f"Logger: {logger.name} (level={logger.level})")
print(f"safe_torch_save available: {callable(safe_torch_save)}")
print(f"extract_state_dict available: {callable(extract_state_dict)}")
print(f"train_one_epoch / validate: {callable(train_one_epoch)} / {callable(validate)}")
print(f"apply_smooth_schedule available: {callable(apply_smooth_schedule)}")
print(f"compute_stage2_scores available: {callable(compute_stage2_scores)}")

---
## 5. utils/ — Losses, Metrics, Priors, Split

### 5a. Losses

In [ ]:
from utils.losses import (
    MultiTaskUncertaintyLoss, SafeCrossEntropyLoss,
    FocalTverskyLoss, SoftDiceLoss, FocalBCELoss,
)

batch, H, W = 2, 32, 32
preds = {
    "tissue": torch.randn(batch, 5, H, W),
    "np": torch.randn(batch, 1, H, W),
    "nc": torch.randn(batch, 10, H, W),
    "hv": torch.randn(batch, 2, H, W),
}
targets = {
    "tissue_sem": torch.randint(0, 5, (batch, H, W), dtype=torch.long),
    "nuclei_nc": torch.randint(0, 10, (batch, H, W), dtype=torch.long),
    "nuclei_np": torch.randint(0, 2, (batch, H, W), dtype=torch.long),
    "nuclei_hv": torch.randn(batch, 2, H, W),
}

criterion = MultiTaskUncertaintyLoss()
total, branch_losses = criterion(preds, targets)
print(f"MultiTaskUncertaintyLoss: total={total.item():.4f}, branches={branch_losses}")

# Individual losses
ce = SafeCrossEntropyLoss()(torch.randn(2, 5, H, W), targets["tissue_sem"])
ft = FocalTverskyLoss()(torch.randn(2, 5, H, W), targets["tissue_sem"])
print(f"SafeCE: {ce.item():.4f}, FocalTversky: {ft.item():.4f}")

# SoftDiceLoss on nuclei probability map
dice = SoftDiceLoss()(torch.randn(2, 1, H, W), targets["nuclei_np"].float())
print(f"SoftDiceLoss: {dice.item():.4f}")

# FocalBCELoss on nuclei probability map
fbce = FocalBCELoss()(torch.randn(2, 1, H, W), targets["nuclei_np"].float())
print(f"FocalBCELoss: {fbce.item():.4f}")

# Demonstrate focal_tversky_weight schedule
criterion2 = MultiTaskUncertaintyLoss()
print(f"\nInitial focal_tversky_weight: {criterion2.focal_tversky_weight}")
criterion2.set_focal_tversky_weight(0.5)
print(f"After set_focal_tversky_weight(0.5): {criterion2.focal_tversky_weight}")
total2, branch_losses2 = criterion2(preds, targets)
print(f"  With focal Tversky: total={total2.item():.4f}, branches={branch_losses2}")
criterion2.switch_to_focal_tversky()
print(f"After switch_to_focal_tversky(): {criterion2.focal_tversky_weight}")
total3, branch_losses3 = criterion2(preds, targets)
print(f"  Full focal Tversky: total={total3.item():.4f}, branches={branch_losses3}")

### 5b. Metrics — Single-batch & Accumulator

In [ ]:
from utils.metrics import PUMAMetrics, SemanticMetricAccumulator

metrics = PUMAMetrics()
result = metrics.calculate_all_metrics(preds, targets)
print("PUMAMetrics output (single batch):")
for k, v in result.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

### 5b-ii. SemanticMetricAccumulator — Multi-batch accumulation (correct usage)

In real training, `SemanticMetricAccumulator` is used to accumulate predictions across all validation batches before computing final metrics. This avoids biased per-batch averages.

In [ ]:
# Simulate accumulating over multiple batches (the correct validation-time pattern)
tissue_acc = metrics.new_semantic_accumulator(num_classes=5, prefix="tissue", device="cpu")
nuclei_acc = metrics.new_semantic_accumulator(num_classes=10, prefix="nuclei", device="cpu")

torch.manual_seed(42)
for sim_batch in range(8):
    # Simulate improving predictions (higher accuracy in later batches)
    scale = 0.5 + 0.5 * (sim_batch / 7.0)  # 0.5 -> 1.0
    tissue_logits = torch.randn(4, 5, 64, 64) * (1.0 / max(scale, 0.1))
    tissue_target = torch.randint(0, 5, (4, 64, 64))
    # Make later batches more accurate by aligning predictions to targets
    tissue_logits.scatter_(1, tissue_target.unsqueeze(1), scale * 8.0)

    nuclei_logits = torch.randn(4, 10, 64, 64)
    nuclei_target = torch.randint(0, 10, (4, 64, 64))
    nuclei_logits.scatter_(1, nuclei_target.unsqueeze(1), scale * 6.0)

    tissue_acc.update(tissue_logits, tissue_target)
    nuclei_acc.update(nuclei_logits, nuclei_target)

tissue_result = tissue_acc.compute()
nuclei_result = nuclei_acc.compute()

print("Accumulated Tissue Metrics (across 8 simulated batches):")
for k, v in tissue_result.items():
    val = f"{v:.4f}" if isinstance(v, float) and v == v else "NaN"
    print(f"  {k}: {val}")

print("\nAccumulated Nuclei Metrics (across 8 simulated batches):")
for k, v in nuclei_result.items():
    val = f"{v:.4f}" if isinstance(v, float) and v == v else "NaN"
    print(f"  {k}: {val}")

### 5b-iii. Per-Class Dice/IoU Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Tissue Dice/IoU
tissue_dice = [tissue_result.get(f"tissue_dice_{i}", float('nan')) for i in range(5)]
tissue_iou = [tissue_result.get(f"tissue_iou_{i}", float('nan')) for i in range(5)]
x = np.arange(5)
width = 0.35
bars1 = axes[0].bar(x - width/2, tissue_dice, width, label='Dice', color='#4CAF50')
bars2 = axes[0].bar(x + width/2, tissue_iou, width, label='IoU', color='#2196F3')
axes[0].set_xticks(x)
axes[0].set_xticklabels([INTERNAL_TISSUE_ID_TO_NAME[i] for i in range(5)], rotation=30, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Tissue Per-Class Dice & IoU (Accumulated)')
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', alpha=0.3)

# Nuclei Dice/IoU
nuclei_dice = [nuclei_result.get(f"nuclei_dice_{i}", float('nan')) for i in range(10)]
nuclei_iou = [nuclei_result.get(f"nuclei_iou_{i}", float('nan')) for i in range(10)]
x2 = np.arange(10)
nuclei_colors = ['#F44336' if i in RARE_NUCLEI_IDS else '#4CAF50' for i in range(10)]
bars3 = axes[1].bar(x2 - width/2, nuclei_dice, width, label='Dice', color='#4CAF50')
bars4 = axes[1].bar(x2 + width/2, nuclei_iou, width, label='IoU', color='#2196F3')
# Mark rare classes
for i in range(10):
    if i in RARE_NUCLEI_IDS:
        axes[1].annotate('RARE', (i, 0), ha='center', va='bottom', fontsize=6, color='red', fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels([PUMA_NUCLEI_ID_TO_NAME[i] for i in range(10)], rotation=45, ha='right')
axes[1].set_ylabel('Score')
axes[1].set_title('Nuclei Per-Class Dice & IoU (RARE marked in red)')
axes[1].legend()
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 5b-iv. Selection Score & Rare Macro Dice Breakdown

In [ ]:
# Demonstrate PUMAMetrics.calculate_all_metrics with accumulated data
all_metrics = metrics.calculate_all_metrics(preds, targets)
full_result = {}
full_result.update(tissue_result)
full_result.update(nuclei_result)
full_result.update(all_metrics)

print("=== Selection Score Breakdown ===")
avg_tissue = full_result.get('avg_tissue_dice', 0)
avg_nuclei = full_result.get('avg_nuclei_dice', 0)
rare_tissue = full_result.get('rare_tissue_macro_dice', 0)
rare_nuclei = full_result.get('rare_nuclei_macro_dice', 0)
rare_macro = full_result.get('rare_macro_dice', 0)
selection = full_result.get('selection_score', 0)
print(f"  avg_tissue_dice:      {avg_tissue:.4f}")
print(f"  avg_nuclei_dice:      {avg_nuclei:.4f}")
print(f"  rare_tissue_dice:     {rare_tissue:.4f}")
print(f"  rare_nuclei_dice:     {rare_nuclei:.4f}")
print(f"  rare_macro_dice:      {rare_macro:.4f}")
print(f"  selection_score:     {selection:.4f}")
print(f"  (0.20*avg_tissue + 0.25*avg_nuclei + 0.55*rare_macro)")

# Visualize score composition
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
components = ['avg_tissue\n(0.20)', 'avg_nuclei\n(0.25)', 'rare_macro\n(0.55)']
values = [float(avg_tissue), float(avg_nuclei), float(rare_macro)]
weights = [0.20, 0.25, 0.55]
weighted = [v * w for v, w in zip(values, weights)]
bars = ax.bar(components, values, color=['#2196F3', '#4CAF50', '#F44336'], alpha=0.7, label='Raw Dice')
ax.bar(components, weighted, color=['#2196F3', '#4CAF50', '#F44336'], alpha=0.4, hatch='//', label='Weighted contribution')
ax.axhline(y=selection, color='black', linestyle='--', label=f'Selection score = {selection:.4f}')
ax.set_ylabel('Dice Score')
ax.set_title('PUMA Selection Score Composition')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5c. Spatial Prior & SC-DFA

In [ ]:
from utils.priors import SpatialLogitAdjuster
from utils.sc_dfa import SCDFA

# SC-DFA
sc_dfa = SCDFA(num_tissue_classes=5, num_nuclei_classes=10)
tissue_logits = torch.randn(1, 5, 32, 32)
nc_bias = sc_dfa(tissue_logits)
print(f"SC-DFA bias shape: {nc_bias.shape}")

# SpatialLogitAdjuster
prior = SpatialLogitAdjuster(num_tissue_classes=5, num_nuclei_classes=10)
nc_logits = torch.randn(1, 10, 32, 32)
adjusted = prior(nc_logits, tissue_logits, "metastatic", lambda_scale=0.2)
print(f"SpatialPrior adjusted shape: {adjusted.shape}")

### 5d. Split Utils & Normalization

In [ ]:
from utils.split_utils import make_or_load_group_split
from utils.normalization import normalize_image

# Normalization demo
dummy_img = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
normed = normalize_image(dummy_img)
print(f"Normalized image shape: {normed.shape}, dtype: {normed.dtype}")
print(f"  mean: {normed.mean():.4f}, std: {normed.std():.4f}")

# Demonstrate make_or_load_group_split with synthetic data
import tempfile, os
source_names = [f"roi_{i:02d}" for i in range(20)]
is_original = [True] * 20
with tempfile.TemporaryDirectory() as tmpdir:
    split_path = os.path.join(tmpdir, "test_split.npz")
    train_idx, val_idx = make_or_load_group_split(
        source_names=source_names,
        is_original=is_original,
        split_path=split_path,
        seed=42,
        train_fraction=0.8,
        force_new=False,
        val_original_only=True,
    )
    print(f"\nmake_or_load_group_split: train={len(train_idx)}, val={len(val_idx)}")
    print(f"  Train indices: {train_idx[:5]}...")
    print(f"  Val indices: {val_idx[:5]}...")
    # Load the saved split to verify
    loaded = np.load(split_path, allow_pickle=True)
    print(f"  Split type: {str(loaded['split_type'])}")
    print(f"  Dataset size: {loaded['dataset_size'].item()}")
    print(f"  Group count: {loaded['group_count'].item()}")

---
## 6. Simulated Training & Metrics Tracking

This section demonstrates the full training workflow pattern used in `stage1_trainer.py`, including:
- Smooth schedule application (`apply_smooth_schedule`)
- Multi-batch metric accumulation with `SemanticMetricAccumulator`
- Loss/metric visualization over epochs

In [ ]:
# Simulate a mini training loop tracing the schedule + loss curves
# This mirrors how stage1_trainer.py uses apply_smooth_schedule + PUMAMetrics

from utils.losses import MultiTaskUncertaintyLoss
from utils.metrics import PUMAMetrics

n_epochs = 30

# Track per-epoch data
history = {
    "epoch": [], "train_loss": [], "val_loss": [],
    "focal_w": [], "sc_dfa_w": [], "prior_w": [],
    "avg_tissue_dice": [], "avg_nuclei_dice": [], "rare_macro_dice": [], "selection_score": [],
    "loss_tissue": [], "loss_np": [], "loss_nc": [], "loss_hv": [],
}

criterion_sim = MultiTaskUncertaintyLoss()
metrics_sim = PUMAMetrics()

torch.manual_seed(99)
for epoch in range(1, n_epochs + 1):
    # Simulate schedule
    focal_w = linear_ramp(epoch, STAGE1_DEFAULT_CONFIG.focal_start_epoch, STAGE1_DEFAULT_CONFIG.focal_full_epoch, STAGE1_DEFAULT_CONFIG.focal_max_weight)
    sc_dfa_w = linear_ramp(epoch, STAGE1_DEFAULT_CONFIG.sc_dfa_start_epoch, STAGE1_DEFAULT_CONFIG.sc_dfa_full_epoch, STAGE1_DEFAULT_CONFIG.sc_dfa_max_weight)
    prior_w = linear_ramp(epoch, STAGE1_DEFAULT_CONFIG.prior_start_epoch, STAGE1_DEFAULT_CONFIG.prior_full_epoch, STAGE1_DEFAULT_CONFIG.prior_max_weight)

    criterion_sim.set_focal_tversky_weight(focal_w)

    # Simulate improving predictions over epochs (model gets better)
    scale = 0.3 + 0.7 * min(epoch / n_epochs, 1.0)
    batch = 2
    H = W = 32
    preds_sim = {
        "tissue": torch.randn(batch, 5, H, W),
        "np": torch.randn(batch, 1, H, W),
        "nc": torch.randn(batch, 10, H, W),
        "hv": torch.randn(batch, 2, H, W),
    }
    # Align predictions to targets more strongly as epochs progress
    targets_sim = {
        "tissue_sem": torch.randint(0, 5, (batch, H, W), dtype=torch.long),
        "nuclei_nc": torch.randint(0, 10, (batch, H, W), dtype=torch.long),
        "nuclei_np": torch.randint(0, 2, (batch, H, W), dtype=torch.long),
        "nuclei_hv": torch.randn(batch, 2, H, W),
    }
    preds_sim["tissue"].scatter_(1, targets_sim["tissue_sem"].unsqueeze(1), scale * 6.0)
    preds_sim["nc"].scatter_(1, targets_sim["nuclei_nc"].unsqueeze(1), scale * 5.0)
    preds_sim["np"] = preds_sim["np"] * (1 - scale * 0.5) + scale * 2.0 * targets_sim["nuclei_np"].unsqueeze(1).float()

    loss_val, branch_losses = criterion_sim(preds_sim, targets_sim)
    metrics_result = metrics_sim.calculate_all_metrics(preds_sim, targets_sim)

    history["epoch"].append(epoch)
    history["train_loss"].append(loss_val.item())
    history["val_loss"].append(loss_val.item() * 0.85 + np.random.randn() * 0.1)
    history["focal_w"].append(focal_w)
    history["sc_dfa_w"].append(sc_dfa_w)
    history["prior_w"].append(prior_w)
    history["avg_tissue_dice"].append(metrics_result["avg_tissue_dice"])
    history["avg_nuclei_dice"].append(metrics_result["avg_nuclei_dice"])
    history["rare_macro_dice"].append(metrics_result["rare_macro_dice"])
    history["selection_score"].append(metrics_result["selection_score"])
    history["loss_tissue"].append(branch_losses[0])
    history["loss_np"].append(branch_losses[1])
    history["loss_nc"].append(branch_losses[2])
    history["loss_hv"].append(branch_losses[3])

print(f"Simulated {n_epochs} epochs. Final selection_score: {history['selection_score'][-1]:.4f}")

### 6b. Training Loss Curves & Schedule Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Loss curves
ax = axes[0, 0]
ax.plot(history["epoch"], history["train_loss"], label="Train loss", linewidth=1.5)
ax.plot(history["epoch"], history["val_loss"], label="Val loss", linewidth=1.5, linestyle='--')
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(alpha=0.3)

# 2. Branch losses
ax = axes[0, 1]
ax.plot(history["epoch"], history["loss_tissue"], label="Tissue CE", linewidth=1.5)
ax.plot(history["epoch"], history["loss_np"], label="Nuclei Prob", linewidth=1.5)
ax.plot(history["epoch"], history["loss_nc"], label="Nuclei CE", linewidth=1.5)
ax.plot(history["epoch"], history["loss_hv"], label="HV L1", linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Branch Losses")
ax.legend()
ax.grid(alpha=0.3)

# 3. Schedule weights
ax = axes[1, 0]
ax.plot(history["epoch"], history["focal_w"], label="FocalTversky weight", linewidth=2)
ax.plot(history["epoch"], history["sc_dfa_w"], label="SC-DFA lambda", linewidth=2)
ax.plot(history["epoch"], history["prior_w"], label="Spatial Prior lambda", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Weight")
ax.set_title("Smooth Schedule (Focal, SC-DFA, Prior)")
ax.legend()
ax.grid(alpha=0.3)

# 4. Selection score & rare dice
ax = axes[1, 1]
ax.plot(history["epoch"], history["selection_score"], label="Selection score", linewidth=2, color='black')
ax.plot(history["epoch"], history["avg_tissue_dice"], label="avg_tissue_dice", linewidth=1.5)
ax.plot(history["epoch"], history["avg_nuclei_dice"], label="avg_nuclei_dice", linewidth=1.5)
ax.plot(history["epoch"], history["rare_macro_dice"], label="rare_macro_dice", linewidth=2, color='red')
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_title("Selection Score & Rare-Focused Metrics")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Stage 2 Refiner — Score Comparison & Visualization

### 7a. Alpha & Keep-Lambda Schedules

In [ ]:
s2_epochs = range(1, STAGE2_DEFAULT_CONFIG.epochs + 1)
alpha_vals = [alpha_schedule(e) for e in s2_epochs]
keep_vals = [keep_lambda_schedule(e) for e in s2_epochs]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(s2_epochs, alpha_vals, label='alpha (residual scale)', linewidth=2, color='#E91E63')
ax.plot(s2_epochs, keep_vals, label='keep_lambda (KD)', linewidth=2, color='#9C27B0')
ax.set_xlabel('Epoch')
ax.set_ylabel('Value')
ax.set_title('Stage 2 Alpha & Keep-Lambda Schedules')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Stage 2: alpha_end={STAGE2_DEFAULT_CONFIG.alpha_end}, keep_lambda_end={STAGE2_DEFAULT_CONFIG.keep_lambda_end}")
print(f"Stage 2: alpha range: [{alpha_vals[0]:.4f}, {alpha_vals[-1]:.4f}]")
print(f"Stage 2: keep range: [{keep_vals[0]:.4f}, {keep_vals[-1]:.4f}]")

### 7b. Stage 2 Score Computation — S1 vs S2 Improvement

In [ ]:
# Simulate Stage 1 baseline and Stage 2 refined nuclei metrics
# This mirrors compute_stage2_scores from stage2_trainer.py
torch.manual_seed(42)

# Simulate S1 metrics across 10 nuclei classes (simulating validation accumulation)
s1_acc = metrics_sim.new_semantic_accumulator(10, "s1_nuclei", IGNORE_INDEX, device="cpu")
s2_acc = metrics_sim.new_semantic_accumulator(10, "s2_nuclei", IGNORE_INDEX, device="cpu")

for _ in range(6):  # simulate 6 val batches
    targets_nc = torch.randint(0, 10, (4, 64, 64))
    # S1 logits (moderate accuracy)
    s1_logits = torch.randn(4, 10, 64, 64)
    s1_logits.scatter_(1, targets_nc.unsqueeze(1), 4.0)
    # S2 logits (improved accuracy, especially for rare classes)
    s2_logits = s1_logits.clone()
    # Boost rare classes in S2
    for rare_cls in [2, 4, 5, 8, 9]:
        rare_mask = (targets_nc == rare_cls)
        if rare_mask.any():
            s2_logits[:, rare_cls] += 3.0 * rare_mask.float()

    s1_acc.update(s1_logits, targets_nc)
    s2_acc.update(s2_logits, targets_nc)

s1_metrics = s1_acc.compute()
s2_metrics = s2_acc.compute()

s2_scores = compute_stage2_scores(metrics_sim, s1_metrics, s2_metrics)

print("=== Stage 2 Score Computation ===")
print(f"S1 macro Dice:  {s2_scores['s1_macro_dice']:.4f}")
print(f"S2 macro Dice:  {s2_scores['s2_macro_dice']:.4f}")
print(f"S1 rare Dice:   {s2_scores['s1_rare_macro_dice']:.4f}")
print(f"S2 rare Dice:   {s2_scores['s2_rare_macro_dice']:.4f}")
print(f"Selection:      {s2_scores['selection_score']:.4f}")
print(f"Improvement:    {s2_scores['improvement_score']:.4f}")
print(f"Beats Stage 1:  {s2_scores['beats_stage1']}")

### 7c. S1 vs S2 Per-Class Nuclei Dice Comparison

In [ ]:
s1_dice_per_class = [s1_metrics.get(f"s1_nuclei_dice_{k}", float('nan')) for k in range(10)]
s2_dice_per_class = [s2_metrics.get(f"s2_nuclei_dice_{k}", float('nan')) for k in range(10)]
delta = [s2 - s1 if not (np.isnan(s2) or np.isnan(s1)) else float('nan')
         for s1, s2 in zip(s1_dice_per_class, s2_dice_per_class)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: grouped bar chart
x = np.arange(10)
width = 0.35
axes[0].bar(x - width/2, s1_dice_per_class, width, label='Stage 1', color='#2196F3', alpha=0.8)
axes[0].bar(x + width/2, s2_dice_per_class, width, label='Stage 2', color='#4CAF50', alpha=0.8)
for i in range(10):
    if i in RARE_NUCLEI_IDS:
        axes[0].annotate('RARE', (i, 0), ha='center', va='bottom', fontsize=5, color='red', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([PUMA_NUCLEI_ID_TO_NAME[i] for i in range(10)], rotation=45, ha='right')
axes[0].set_ylabel('Dice Score')
axes[0].set_title('Stage 1 vs Stage 2 Per-Class Nuclei Dice')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Right: improvement delta
colors = ['#4CAF50' if d >= 0 else '#F44336' if not np.isnan(d) else '#9E9E9E' for d in delta]
axes[1].bar(x, delta, color=colors)
axes[1].axhline(y=0, color='black', linewidth=0.5)
for i in range(10):
    if i in RARE_NUCLEI_IDS:
        axes[1].annotate('RARE', (i, 0), ha='center', va='top', fontsize=5, color='red', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([PUMA_NUCLEI_ID_TO_NAME[i] for i in range(10)], rotation=45, ha='right')
axes[1].set_ylabel('Dice Improvement (S2 - S1)')
axes[1].set_title('Stage 2 Improvement Over Stage 1 (green=improved, red=regressed)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. inference/ — WSI Inference Pipeline

In [ ]:
from inference import main as inference_main
from inference.tiling import (
    find_single_tif, read_rgb_uint8, normalize_tile,
    make_tile_starts, pad_reflect, autocast_enabled,
)
from inference.model_loader import load_stage1, load_stage2
from inference.site_classifier import load_site_classifier, predict_site_type, resolve_site_type
from inference.cellpose_flow import CellposeFlowGenerator as InferenceCellposeFlowGenerator
from inference.postprocessing import (
    hv_instance_segmentation, classify_instances, instances_to_polygons,
)
from inference.infer_wsi import DEFAULT_STAGE1_CP as INFER_STAGE1_CP

print(f"Inference main: {inference_main}")
print(f"make_tile_starts for len=2000, tile=1024, stride=768:"
      f" {make_tile_starts(2000, 1024, 768)}")

# pad_reflect demo
small_tile = np.random.randint(0, 256, (512, 768, 3), dtype=np.uint8)
padded, rh, rw = pad_reflect(small_tile, 1024)
print(f"pad_reflect: {small_tile.shape} -> {padded.shape} (real_h={rh}, real_w={rw})")

---
## 9. scripts/ — Entry Points

In [ ]:
from scripts.run_preprocess import main as script_preprocess
from scripts.run_stage1 import main as script_stage1
from scripts.run_stage2 import main as script_stage2
from scripts.run_inference import main as script_inference

print("All script entry points imported:")
print(f"  run_preprocess: {script_preprocess}")
print(f"  run_stage1:     {script_stage1}")
print(f"  run_stage2:     {script_stage2}")
print(f"  run_inference:  {script_inference}")

---
## 10. Verify All Exports from __init__.py Files

In [ ]:
print("=== data.__init__ ===")
from data import (
    HV_GRAD_THRESHOLD, INTERNAL_TISSUE_ID_TO_NAME, LOSS_MULTIPLIERS,
    NORMALIZATION_MEAN, NORMALIZATION_STD, NUCLEI_CLASS_WEIGHTS,
    NUM_NUCLEI_CLASSES, NUM_TISSUE_CLASSES,
    PUMA_NUCLEI_ID_TO_NAME, PUMA_NUCLEI_NAME_TO_ID,
    PUMA_TISSUE_ID_TO_NAME, PUMA_TISSUE_NAME_TO_ID,
    RARE_NUCLEI_IDS, RARE_NUCLEI_SAMPLE_BONUS,
    RARE_TISSUE_IDS, RARE_TISSUE_IDS_PUMA, RARE_TISSUE_SAMPLE_BONUS,
    STAGE2_NUCLEI_WEIGHTS, TISSUE_CLASS_WEIGHTS,
)
print("  All data constants imported OK")

print("\n=== models.__init__ ===")
from models import build_cnn_backbone, UnifiedPanopticNet, ResidualNucleiRefinerUNet, build_stage2_input
print("  All models imported OK")

print("\n=== utils.__init__ ===")
from utils import MultiTaskUncertaintyLoss, PUMAMetrics, SpatialLogitAdjuster, SCDFA
print("  All utils imported OK")

print("\n=== training.__init__ ===")
from training import extract_state_dict, load_large_checkpoint, safe_torch_save, logger, setup_logger, stage1_main, stage2_main, train_one_epoch, validate
print("  All training imported OK")

print("\n=== All modules imported successfully ===")

---
## 11. Data Leakage Prevention Verification

This project uses **group-based splitting** by source image to prevent data leakage. Rare-centered crops derived from the same source image are forced into the same split side. Validation uses only original 1024×1024 samples (`val_original_only=True`).

In [ ]:
from configs import STAGE1_DEFAULT_CONFIG, STAGE2_DEFAULT_CONFIG, PATHS
from pathlib import Path

print("=== Data Leakage Prevention Checks ===")
print(f"  Split method:        GROUP-BASED (by source image)")
print(f"  Split file:          {PATHS.split_file}")
print(f"  val_original_only:   {STAGE1_DEFAULT_CONFIG.val_original_only}")
print(f"  force_new_split:     {STAGE1_DEFAULT_CONFIG.force_new_split}")
print(f"  val_ratio:           {STAGE1_DEFAULT_CONFIG.val_ratio}")
print(f"  Stage 2 val_orig:   {STAGE2_DEFAULT_CONFIG.val_original_only}")
print()
if PATHS.split_file.exists():
    import numpy as np
    data = np.load(PATHS.split_file, allow_pickle=True)
    split_type = str(data['split_type'])
    train_src = set(str(x) for x in data['train_sources'])
    val_src = set(str(x) for x in data['val_sources'])
    overlap = train_src & val_src
    print(f"  Existing split:      {split_type}")
    print(f"  Train groups:        {len(train_src)}")
    print(f"  Val groups:          {len(val_src)}")
    print(f"  Group overlap:       {len(overlap)} (should be 0)")
    assert len(overlap) == 0, "CRITICAL: Source group overlap detected! Data leakage!"
    train_count = int(data['train_indices'].shape[0])
    val_count = int(data['val_indices'].shape[0])
    print(f"  Train samples:       {train_count}")
    print(f"  Val samples:         {val_count}")
    print("  >>> LEAKAGE CHECK: PASS (no source group overlap)" if len(overlap) == 0 else "  >>> FAIL")
else:
    print("  No split file exists yet. Will be created on first training run.")
    print("  The split uses group-based strategy (leakage-safe by design).")
print()
print("Rare-focused strategy:")
print("  - Rare-centered crops (augmented) are ONLY assigned to training")
print("  - Validation contains ONLY original 1024x1024 samples")
print("  - Sample weights up the rare classes in the WeightedRandomSampler")

---
## 12. Run Stage 1 Training — UnifiedPanopticNet

This cell runs the actual Stage 1 training. After training completes, the best and last checkpoints are **automatically saved** to `checkpoints/`:
- `puma_epoch_best_s1.pth` (best validation selection score)
- `puma_epoch_last_s1.pth` (last epoch)

> ⚠️ Training takes ~2–3 days on G4 (A100 80GB). Click ▶ if you are ready.

In [ ]:
# ──────────────────────────────────────────────────────
# Stage 1 Training — UnifiedPanopticNet
# """
# WARNING: Full training takes ~2-3 days.
# Uncomment the line below when you are ready:
# ──────────────────────────────────────────────────────

from training import stage1_main
from configs import PATHS

# Check dataset exists before starting
data_dir = PATHS.data_dir
has_tissue = (data_dir / "tissue_sem").is_dir()
has_nuclei = (data_dir / "nuclei_nc").is_dir()
has_images = (data_dir / "images").is_dir()

print("=== Stage 1 Pre-flight Check ===")
print(f"  Data dir:            {data_dir}")
print(f"  tissue_sem/ exists:  {has_tissue}")
print(f"  nuclei_nc/ exists:   {has_nuclei}")
print(f"  images/ exists:      {has_images}")
print(f"  Checkpoint dir:      {PATHS.checkpoint_dir}")
print()

if not (has_tissue and has_nuclei and has_images):
    raise RuntimeError(
        "Processed dataset not found. Run preprocessing first, or mount the correct "
        "Drive path. See Section 0 for Drive setup."
    )

print("Dataset ready. Launching Stage 1 training...")
print("NOTE: To actually run training, call stage1_main() below:")
print()
# ── Uncomment to launch ──
# stage1_main()

print("[DRY RUN] Stage 1 would start now. Uncomment stage1_main() to launch.")

### 12a. Verify Stage 1 Checkpoints

After Stage 1 completes, verify the saved checkpoints exist and are loadable.

In [ ]:
from training.checkpoint import safe_torch_save, load_large_checkpoint, extract_state_dict
from pathlib import Path

s1_best = PATHS.checkpoint_dir / "puma_epoch_best_s1.pth"
s1_last = PATHS.checkpoint_dir / "puma_epoch_last_s1.pth"

print("=== Stage 1 Checkpoint Verification ===")
for ckpt_path, label in [(s1_best, "Best"), (s1_last, "Last")]:
    if ckpt_path.exists():
        size_mb = ckpt_path.stat().st_size / 1e6
        print(f"  [{label}] {ckpt_path.name}: {size_mb:.1f} MB")
        try:
            ckpt = load_large_checkpoint(ckpt_path, map_location="cpu")
            epoch = ckpt.get("epoch", "?")
            score = ckpt.get("best_score", "?")
            state_keys = list(ckpt.get("model_state", {}).keys())
            print(f"    epoch={epoch}, best_score={score}")
            print(f"    model_state keys: {len(state_keys)} parameters")
            print(f"    -> Checkpoint valid and loadable.")
        except Exception as e:
            print(f"    -> CORRUPTED: {e}")
    else:
        print(f"  [{label}] {ckpt_path.name}: NOT FOUND (run Stage 1 first)")

---
## 13. Run Stage 2 Training — Residual Nuclei Refiner

Stage 2 requires the Stage 1 checkpoint (`puma_epoch_best_s1.pth`). It refines nuclei classification for rare classes.

After training completes, checkpoints are saved:
- `nuclei_refiner_residual_best.pth` (best improvement over Stage 1)
- `nuclei_refiner_residual_last.pth` (last epoch)

In [ ]:
# ──────────────────────────────────────────────────────
# Stage 2 Training — Residual Nuclei Refiner UNet
# """
# WARNING: Full training takes ~6-12 hours.
# Requires Stage 1 checkpoint to exist.
# Uncomment the line below when you are ready:
# ──────────────────────────────────────────────────────

from training import stage2_main

s1_best = PATHS.checkpoint_dir / "puma_epoch_best_s1.pth"

print("=== Stage 2 Pre-flight Check ===")
print(f"  Stage 1 checkpoint:  {s1_best}")
print(f"  Stage 1 exists:      {s1_best.exists()}")
print(f"  Checkpoint dir:      {PATHS.checkpoint_dir}")
print()

if not s1_best.exists():
    print("  -> Stage 1 checkpoint not found. Run Stage 1 training (Section 12) first.")
    print()
    print("[DRY RUN] Stage 2 would start now. Uncomment stage2_main() to launch.")
else:
    print("Stage 1 checkpoint ready. Launching Stage 2 training...")
    print("NOTE: To actually run training, call stage2_main() below:")
    print()
    # ── Uncomment to launch ──
    # stage2_main()
    print("[DRY RUN] Stage 2 would start now. Uncomment stage2_main() to launch.")

### 13a. Verify Stage 2 Checkpoints

In [ ]:
s2_best = PATHS.checkpoint_dir / "nuclei_refiner_residual_best.pth"
s2_last = PATHS.checkpoint_dir / "nuclei_refiner_residual_last.pth"

print("=== Stage 2 Checkpoint Verification ===")
for ckpt_path, label in [(s2_best, "Best"), (s2_last, "Last")]:
    if ckpt_path.exists():
        size_mb = ckpt_path.stat().st_size / 1e6
        print(f"  [{label}] {ckpt_path.name}: {size_mb:.1f} MB")
        try:
            ckpt = load_large_checkpoint(ckpt_path, map_location="cpu")
            epoch = ckpt.get("epoch", "?")
            score = ckpt.get("selection_score", "?")
            beats = ckpt.get("beats_stage1", "?")
            print(f"    epoch={epoch}, selection_score={score}, beats_stage1={beats}")
            print(f"    -> Checkpoint valid and loadable.")
        except Exception as e:
            print(f"    -> CORRUPTED: {e}")
    else:
        print(f"  [{label}] {ckpt_path.name}: NOT FOUND (run Stage 2 first)")

### 13b. Save Final Combined Checkpoint & Summary

Bundle Stage 1 + Stage 2 models together for deployment.

In [ ]:
import json, datetime

print("=" * 60)
print("  SymbioPan Training Complete — Save Summary")
print(f"  Finished at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

checkpoints_found = []
for name in ["puma_epoch_best_s1.pth", "puma_epoch_last_s1.pth",
             "nuclei_refiner_residual_best.pth", "nuclei_refiner_residual_last.pth"]:
    p = PATHS.checkpoint_dir / name
    if p.exists():
        checkpoints_found.append({"name": name, "size_mb": round(p.stat().st_size / 1e6, 2)})
        print(f"  ✓ {name:40s}  {p.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"  ✗ {name:40s}  NOT FOUND")

if not checkpoints_found:
    print("\n  No checkpoints found. Run training first (Sections 12-13).")
else:
    # Save a summary manifest
    manifest = {
        "pipeline": "SymbioPan PUMA Track 2",
        "finished": datetime.datetime.now().isoformat(),
        "checkpoints": checkpoints_found,
        "checkpoint_dir": str(PATHS.checkpoint_dir),
        "data_dir": str(PATHS.data_dir),
        "stage1_config": {
            "batch_size": S1C.batch_size,
            "epochs": S1C.epochs,
            "focal_max_weight": S1C.focal_max_weight,
            "sc_dfa_max_weight": S1C.sc_dfa_max_weight,
            "prior_max_weight": S1C.prior_max_weight,
            "multi_gpu": S1C.multi_gpu,
        },
        "stage2_config": {
            "batch_size": S2C.batch_size,
            "epochs": S2C.epochs,
            "alpha_end": S2C.alpha_end,
            "keep_lambda_end": S2C.keep_lambda_end,
        },
        "leakage_prevention": {
            "split": "group_based",
            "val_original_only": True,
        },
    }
    manifest_path = PATHS.checkpoint_dir / "training_manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\n  Manifest saved: {manifest_path}")
    print("  All models saved successfully. Notebook complete.")